# IPL Batting Analysis

Dataset: IPL ball-by-ball data (2008–2026)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

## 1. Load data

In [ ]:
df = pd.read_csv('IPL.csv', low_memory=False)
print(f"Dataset shape: {df.shape}")
print(f"Seasons covered: {sorted(df['season'].dropna().unique())}")

## 2. Total runs per batsman

In [ ]:
total_runs = df.groupby('batter')['runs_batter'].sum().sort_values(ascending=False)
print("Top 10 Run Scorers:")
total_runs.head(10)

## 3. Dismissals

In [ ]:
dismissals = (
    df[df['player_out'].notna() & (df['player_out'] != '')]
    .groupby('player_out')
    .size()
    .rename('dismissals')
)
dismissals.head()

## 4. Batting average (raw, unfiltered)

Average = total runs / times dismissed. A separate qualified view (min 20 dismissals) is used only for the Top 10 below.

In [ ]:
batting_avg_raw = (total_runs / dismissals).dropna().rename('batting_avg')

batting_avg_qualified = batting_avg_raw[dismissals >= 20].sort_values(ascending=False)
print("Top 10 Batting Averages (min 20 dismissals):")
batting_avg_qualified.head(10).round(2)

## 5. Strike rate (raw, unfiltered)

Strike rate = (runs / balls faced) * 100. Uses `balls_faced` (excludes wides only — the correct cricket convention), **not** `valid_ball` (which also excludes no-balls and over-states strike rate).

In [ ]:
balls_faced = df.groupby('batter')['balls_faced'].sum().rename('balls_faced')
strike_rate_raw = ((total_runs / balls_faced) * 100).rename('strike_rate')

strike_rate_qualified = strike_rate_raw[balls_faced >= 200].sort_values(ascending=False)
print("Top 10 Strike Rates (min 200 balls faced):")
strike_rate_qualified.head(10).round(2)

## 6. Innings played

Counts distinct `(match_id, innings)` pairs so Super Over appearances are counted as separate innings rather than merged into the main match.

In [ ]:
innings = (
    df.assign(_match_innings=df['match_id'].astype(str) + '_' + df['innings'].astype(str))
    .groupby('batter')['_match_innings']
    .nunique()
    .rename('innings')
)
innings.head()

## 7. Combined stats table

In [ ]:
stats_df = pd.DataFrame({
    'batter': total_runs.index,
    'total_runs': total_runs.values
})
stats_df = (
    stats_df
    .merge(batting_avg_raw, left_on='batter', right_index=True, how='left')
    .merge(strike_rate_raw, left_on='batter', right_index=True, how='left')
    .merge(innings, left_on='batter', right_index=True, how='left')
    .merge(dismissals, left_on='batter', right_index=True, how='left')
    .merge(balls_faced, left_on='batter', right_index=True, how='left')
)

# Most played season per batter
season_data = (
    df.groupby('batter')['year']
    .agg(lambda x: x.mode()[0])
    .reset_index()
    .rename(columns={'year': 'most_active_year'})
)
stats_df = stats_df.merge(season_data, on='batter', how='left')

stats_df.head()

In [ ]:
stats_df.to_csv('ipl_batting_stats_v2.csv', index=False)
print(f"Saved ipl_batting_stats_v2.csv — {len(stats_df)} batters")

## 8. Visualizations

In [ ]:
sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
fig.suptitle('IPL Batting Analysis Dashboard', fontsize=18, fontweight='bold', y=1.02)

# Chart 1 — Top 10 Run Scorers
top10_runs = total_runs.head(10)
axes[0].barh(top10_runs.index[::-1], top10_runs.values[::-1], color='steelblue')
axes[0].set_title('Top 10 Run Scorers', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Total Runs')

# Chart 2 — Top 10 Batting Averages (qualified)
top10_avg = batting_avg_qualified.head(10)
axes[1].barh(top10_avg.index[::-1], top10_avg.values[::-1], color='seagreen')
axes[1].set_title('Top 10 Batting Averages\n(min 20 dismissals)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Batting Average')

# Chart 3 — Top 10 Strike Rates (qualified)
top10_sr = strike_rate_qualified.head(10)
axes[2].barh(top10_sr.index[::-1], top10_sr.values[::-1], color='darkorange')
axes[2].set_title('Top 10 Strike Rates\n(min 200 balls)', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Strike Rate')

plt.tight_layout()
plt.savefig('ipl_batting_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()